In [ ]:
import random
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


#this snippet contains important object classes, including nodes and bots

class manbot():
    def __init__(self, name, coop_stat):
        self.name = name


        self.coop_stat = coop_stat

    def __repr__(self):
        return f"manbot(name={self.name}, coop_stat={self.coop_stat})"



class point():
    def __init__(self, xpos, ypos):
        self.xpos = xpos
        self.ypos = ypos

    def __repr__(self):
        return f"node(xpos={self.xpos:.2f}, ypos={self.ypos:.2f})"


In [ ]:
class mapper():
    def __init__(self, repete=20, grid=100):
        self.grid = grid
        self.node_dic = {}

        print("(1). Random Network (2). Square Grid (3). Linear Network (4). Double Spanning Tree (5). ODE Single Node")

        self.layout_choice = int(input("Layout Choice: "))

        if self.layout_choice in [1,4]:
            i = 0
            while i < repete: 
                xx = random.uniform(0, grid)
                yy = random.uniform(0, grid)
                new = point(xx, yy)
                if not any(abs(n.xpos - xx) < 1e-5 and abs(n.ypos - yy) < 1e-5 for n in self.node_dic):
                    self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                    i += 1

        if self.layout_choice == 2:
            i = 0
            while i < grid:
                y = 0
                while y <= grid:
                    new = point(i,y)
                    self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                    y += 1
                i += 1


        

class connector():
    def __init__(self, node_dic, av=25.0):
        self.node_dic = node_dic

        if my_mapper.layout_choice == 1:
            for node1 in self.node_dic:
                for node2 in self.node_dic:
                    if node1 != node2:
                        dist = (((node1.xpos - node2.xpos) ** 2) + ((node1.ypos - node2.ypos) ** 2)) ** 0.5
                        if dist <= av:
                            if node2 not in self.node_dic[node1]['connections']:
                                self.node_dic[node1]['connections'].append(node2)
                                self.node_dic[node2]['connections'].append(node1)

            for node1 in self.node_dic:
                if len(self.node_dic[node1]['connections']) == 0:
                    closest_node = None
                    closest_dist = float('inf')
                    for node2 in self.node_dic:
                        if node1 != node2:
                            dist = (((node1.xpos - node2.xpos) ** 2) + ((node1.ypos - node2.ypos) ** 2)) ** 0.5
                            if dist < closest_dist:
                                closest_dist = dist
                                closest_node = node2
                    if closest_node:
                        self.node_dic[node1]['connections'].append(closest_node)
                        if node1 not in self.node_dic[closest_node]['connections']:
                            self.node_dic[closest_node]['connections'].append(node1)
    
            G = nx.Graph()
            for n in self.node_dic:
                G.add_node(n)
            for n in self.node_dic:
                for neighbor in self.node_dic[n]['connections']:
                    G.add_edge(n, neighbor)
    
            while not nx.is_connected(G):
                components = list(nx.connected_components(G))
                if len(components) <= 1:
                    break
                best_pair = None
                best_dist = float('inf')
                for i in range(len(components)):
                    for j in range(i + 1, len(components)):
                        comp1 = components[i]
                        comp2 = components[j]
                        for n1 in comp1:
                            for n2 in comp2:
                                dist = (((n1.xpos - n2.xpos) ** 2) + ((n1.ypos - n2.ypos) ** 2)) ** 0.5
                                if dist < best_dist:
                                    best_dist = dist
                                    best_pair = (n1, n2)
                if best_pair:
                    node1, node2 = best_pair
                    self.node_dic[node1]['connections'].append(node2)
                    self.node_dic[node2]['connections'].append(node1)
                    G.add_edge(node1, node2)



        elif my_mapper.layout_choice == 2:

            # Build a coord -> node lookup so we can find existing grid neighbors.
            # Without this, `node(x, y) in self.nodes.keys()` was always False
            # because a freshly-constructed node has a different identity.
            by_coord = {(n.xpos, n.ypos): n for n in self.node_dic}

            for n in self.node_dic:
                left  = by_coord.get((n.xpos - 1, n.ypos))
                right = by_coord.get((n.xpos + 1, n.ypos))
                down  = by_coord.get((n.xpos, n.ypos - 1))
                up    = by_coord.get((n.xpos, n.ypos + 1))

                if left is not None:
                    self.node_dic[n]['connections'].append(left)
                if right is not None:
                    self.node_dic[n]['connections'].append(right)
                if down is not None:
                    self.node_dic[n]['connections'].append(down)
                if up is not None:
                    self.node_dic[n]['connections'].append(up)

            G = nx.Graph()
            for n in self.node_dic:
                G.add_node(n)
            for n in self.node_dic:
                for neighbor in self.node_dic[n]['connections']:
                    G.add_edge(n, neighbor)



       
    

        

            

In [ ]:
#this generates a random initial populator

class populator():
    def __init__(self, node_dic, name_count=1):
        self.node_dic = node_dic
        self.name_count = name_count

        for nodd in self.node_dic:
            self.node_dic[nodd]['inhabitants'].clear()
                
                

        for nodd in self.node_dic:
            i = 0 
            while i < 5:
                bot_name = 'Bot' + str(self.name_count)
                
                if random.random() <= 0.65:
                    coop_stat = 0
                else:
                    coop_stat = 1
                    
                new_bot = manbot(bot_name, coop_stat)
    
                self.node_dic[nodd]['inhabitants'].append(new_bot)
                self.name_count += 1
                i += 1


In [ ]:
import random

class play_options():
    def __init__(self, node_dic1, node_dic2, node1, bot1, name_count, capacity, base):


        #name count is just variable used to name bots; ignore
        self.name_count = name_count
        self.node_dic2 = node_dic2

        action_probability = random.random()

        #model as follows: [[CC, CD], [DC, DD]]
        payout_matrix = [[0.6, -0.4],[0.8, -0.3]]

        
        #chooses bots destination IF it moves
        destination = random.choice(self.node_dic2[node1]['connections'])

        base_penalty = base
        over_penalty = 0.0

        #sets overcapacity penalty
        if len(node_dic1[node1]['inhabitants']) > capacity:
            over_penalty = (len(node_dic1[node1]['inhabitants'])-capacity)/(len(node_dic1[node1]['inhabitants']))

        elif len(node_dic1[node1]['inhabitants']) == 1:
            over_penalty = 0.25

        total_penalty = (base_penalty + over_penalty)/3

        #MACRO SCENARIO 1: random chance of moving
        #increases population sizes for both C and D
        if action_probability <= (0.2 + total_penalty):

            self.node_dic2[destination]['inhabitants'].append(bot1)

        #MACRO SCENARIO 2: chance of dying 
        #high chance of dying helps cooperators...?
        elif (0.2 + total_penalty) < action_probability <= (0.1 + 2*(total_penalty)):
            pass

        #MACRO SCENARIO 3: random chance of playing other players, limited to within node
        #Creates more fluctations within C and D behavior/node populations
        elif (0.1 + (2*total_penalty)) < action_probability:

            if len(node_dic1[node1]['inhabitants']) > 1:
                
                bot2 = random.choice(node_dic1[node1]['inhabitants'])

                payout = payout_matrix[bot1.coop_stat][bot2.coop_stat]
                
                #hand of god is the CV that determines the outcome of the payout
                hand_of_god = random.random()


                #positive payout, chance of reproduction
                if payout > 0:
                    self.node_dic2[node1]['inhabitants'].append(bot1)
                    if hand_of_god <= payout:
                        self.name_count += 1
                        new_bot = manbot('Bot' + str(self.name_count), bot1.coop_stat)
                        self.node_dic2[node1]['inhabitants'].append(new_bot)

                #negative payout, chance of death
                elif payout < 0:
                    if hand_of_god <= (-1*payout):
                        pass
                    else:
                        self.node_dic2[node1]['inhabitants'].append(bot1)

            #if the bot went to play but couldn't, it will just survive to next round
            elif len(node_dic1[node1]['inhabitants']) < 2:
                self.node_dic2[node1]['inhabitants'].append(bot1)
                    

                #new way to deal with deaths -- nothing happens, so they aren't transferred to next round!
        

        #MACRO SCENARIO 4: nothing happens
        else:
            self.node_dic2[node1]['inhabitants'].append(bot1)

    def return_dictionary(self):

        return self.node_dic2
    
            


            

            

        

    

In [ ]:
class simulator():
    def __init__(self, generations, carrying_capacity, base_punishment):

        i = 0
        self.master_dictionary = {}

        
        #generation
        while i < generations:
            
            #initiate dictionaries
            if i == 0:
                dictionary1 = my_populator.node_dic
                dictionary2 = {n: {'connections': dictionary1[n]['connections'], 'attributes': dictionary1[n]['attributes'], "inhabitants": []} for n in dictionary1}
                self.master_dictionary.update({i: dictionary1})
        
            else:
                dictionary1 = dictionary2
                dictionary2 = {n: {'connections': dictionary1[n]['connections'], 'attributes': dictionary1[n]['attributes'], "inhabitants": []} for n in dictionary1}
                
            #loops through nodes, then bots
            for node in dictionary1:
                for bot in dictionary1[node]['inhabitants']:
                    my_play_options = play_options(dictionary1, dictionary2, node, bot, my_populator.name_count, carrying_capacity, base_punishment)
                    dictionary2 = my_play_options.return_dictionary()
        
        
            i += 1

            #final recording of generation
            self.master_dictionary.update({i: dictionary2})

    



In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import networkx as nx


class animator():
    def __init__(self, master_dictionary, gridsize=20, frame_count=1):
        self.master = master_dictionary
        self.gridsize = gridsize
        self.frames = list(range(0, len(master_dictionary), frame_count))

        gen0 = master_dictionary[0]
        self.G = nx.Graph()
        self.pos = {n: (n.xpos, n.ypos) for n in gen0}
        for n in gen0:
            self.G.add_node(n)
            for nb in gen0[n]['connections']:
                self.G.add_edge(n, nb)

        self.fig, self.ax = plt.subplots(figsize=(15, 7))

        self.cmap = LinearSegmentedColormap.from_list("RPB", ['#d62728', '#D8BFD8', '#1f77b4'])
        sm = plt.cm.ScalarMappable(cmap=self.cmap, norm=plt.Normalize(0, 1))
        sm.set_array([])
        cbar = self.fig.colorbar(sm, ax=self.ax, fraction=0.04, pad=0.04)
        cbar.set_label('Strategy (Defection ↔ Cooperation)', rotation=270, labelpad=15, fontweight='bold')
        cbar.set_ticks([0, 0.5, 1])
        cbar.set_ticklabels(['Defection', '50/50', 'Cooperation'])

    def _draw(self, frame):
        self.ax.clear()
        nodes = self.master[frame]
        
        live_nodes, live_sizes, live_colors = [], [], []
        empty_nodes = []
        total = 0
        
        for n in self.G.nodes:
            bots = nodes[n]['inhabitants']
            total += len(bots)
            if bots:
                live_nodes.append(n)
                live_sizes.append(50 + len(bots) * 12)
                live_colors.append(sum(1 for b in bots if b.coop_stat != 1) / len(bots))
            else:
                empty_nodes.append(n)

        nx.draw_networkx_edges(self.G, self.pos, ax=self.ax, edge_color='gainsboro', width=1.2)
        
        if live_nodes:
            nx.draw_networkx_nodes(self.G, self.pos, ax=self.ax, nodelist=live_nodes,
                                   node_size=live_sizes, node_color=live_colors,
                                   cmap=self.cmap, vmin=0, vmax=1)
        
        if empty_nodes:
            nx.draw_networkx_nodes(self.G, self.pos, ax=self.ax, nodelist=empty_nodes,
                                   node_size=0, node_color='gray', node_shape='^')

        pad = self.gridsize * 0.05
        self.ax.set_xlim(-pad, self.gridsize + pad)
        self.ax.set_ylim(-pad, self.gridsize + pad)
        self.ax.set_title(f"Iteration: {frame}  |  Population: {total}", fontweight='bold')
        self.ax.axis('off')

    def animate(self):
        ani = FuncAnimation(self.fig, self._draw, frames=self.frames, repeat=False)
        plt.close(self.fig)
        return ani

In [ ]:
import matplotlib.pyplot as plt


class population_plot():
    def __init__(self, master_dictionary):
        self.master = master_dictionary
        
    def plot(self):
        frames = sorted(self.master.keys())
        cooperators = []
        defectors = []
        
        for f in frames:
            c = d = 0
            for n in self.master[f]:
                for bot in self.master[f][n]['inhabitants']:
                    if bot.coop_stat == 1:
                        d += 1
                    else:
                        c += 1
            cooperators.append(c)
            defectors.append(d)
        
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(frames, cooperators, color='#1f77b4', label='Cooperators', linewidth=2)
        ax.plot(frames, defectors, color='#d62728', label='Defectors', linewidth=2)
        ax.set_xlabel('Generation', fontweight='bold')
        ax.set_ylabel('Population', fontweight='bold')
        ax.set_title('Cooperators vs Defectors Over Time', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

In [ ]:
#mapping step:

my_mapper = mapper(repete=2, grid=4)
my_connector = connector(my_mapper.node_dic, av=1.5)
my_populator = populator(my_connector.node_dic)


In [ ]:
#simulation step
my_simulator = simulator(generations = 300, carrying_capacity = 120000, base_punishment = 0.10)

In [ ]:
#visualization step
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 125.0
strategy_animation = animator(my_simulator.master_dictionary, frame_count = 1).animate()
HTML(strategy_animation.to_jshtml())


In [ ]:
population_plot(my_simulator.master_dictionary).plot()